# LearnLoop: audited Qwen3-32B reference run

This notebook runs the dense 32.8B reference **in Colab only**. Select a GPU runtime first. It refuses GPUs below 22 GiB, keeps LearnLoop tools disabled for the reference, and exports hashed raw evidence. Colab availability and paid upgrades are never automatic; a parent should supervise accounts or charges.

In [ ]:
import subprocess
memory = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], text=True)
gpu_mib = max(int(line.strip()) for line in memory.splitlines() if line.strip())
print(f"Largest GPU: {gpu_mib / 1024:.1f} GiB")
assert gpu_mib >= 22528, "Qwen3-32B needs an L4/A100-class GPU (at least 22 GiB). Do not continue on a T4."

In [ ]:
!git clone --depth 1 https://github.com/samveerrana/LearnLoop.git
%cd LearnLoop
!pip -q install -e . --no-deps
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve >/tmp/ollama.log 2>&1 &
import time
time.sleep(4)
!ollama pull qwen3:32b-q4_K_M

In [ ]:
!python -m learnloop.reference_bundle --create qwen3-32b-bundle --suite benchmarks/benchmark-100-v5/suite.json --knowledge-cases benchmarks/fresh-knowledge-v1/cases.json --reference-manifest configs/reference-qwen3-32b.json

In [ ]:
!python -m learnloop.reference_bundle --verify qwen3-32b-bundle
!zip -qr qwen3-32b-bundle.zip qwen3-32b-bundle
from google.colab import files
files.download("qwen3-32b-bundle.zip")